# AutoRewardPlus — Google Colab Runner

Automatically run Microsoft Rewards tasks on Google Colab.

---
### Notes
- **Session resets** when Colab disconnects — you'll need to re-run from the start each time
- **Runs headless** (no visible browser) — enable `debugLogs: true` if troubleshooting
- **Runtime**: ~15-30 min per account depending on search count
- **Colab free tier**: max ~12h runtime, disconnects if idle too long
- **Best for testing**. For production use Docker/VPS as described in the README

---

## Step 1: Install Node.js >= 24

Colab ships with an older Node.js (~18). This script requires **Node.js >= 24**.

In [ ]:
# @title Install Node.js 24
%%bash
echo "=== Installing Node.js 24 ==="
curl -fsSL https://deb.nodesource.com/setup_24.x | bash - > /dev/null 2>&1
apt-get install -y nodejs > /dev/null 2>&1
echo "Node version: $(node -v)"
echo "npm version: $(npm -v)"
node_ver=$(node -v | sed 's/v//' | cut -d. -f1)
if [ "$node_ver" -ge 24 ]; then
  echo "OK: Node.js >= 24"
else
  echo "ERROR: Node.js version too old, got v$node_ver"
  exit 1
fi

## Step 2: Clone source code

In [ ]:
# @title Clone from GitHub
import os

REPO_URL = "https://github.com/nam091/AutoRewardPlus.git"  # @param {type:"string"}
BRANCH = "v3"  # @param {type:"string"}
PROJECT_DIR = "/content/AutoRewardPlus"

if os.path.exists(PROJECT_DIR):
    !rm -rf "{PROJECT_DIR}"

!git clone -b "{BRANCH}" --single-branch "{REPO_URL}" "{PROJECT_DIR}"
%cd "{PROJECT_DIR}"
print(f"\nCloned to {PROJECT_DIR} (branch: {BRANCH})")

## Step 3: Install dependencies + Chromium

In [ ]:
# @title Install npm packages + system deps + Chromium
%%bash
echo "=== Installing system dependencies for Chromium ==="
apt-get update -qq && apt-get install -y -qq \
  libnss3 libnspr4 libatk1.0-0 libatspi2.0-0 libdrm2 libdbus-1-3 \
  libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 \
  libgbm1 libpango-1.0-0 libcairo2 libasound2 libcups2 \
  > /dev/null 2>&1
echo "OK: System libs installed"

echo ""
echo "=== Installing npm dependencies ==="
cd /content/AutoRewardPlus
npm install 2>&1 | tail -5
echo ""

echo "=== Installing Chromium (full) ==="
npx patchright install --with-deps chromium 2>&1 | tail -10
echo ""
echo "OK: Dependencies ready"

## Step 4: Configure accounts

Paste your account list as JSON below. Supports **multiple accounts** with individual proxy settings.

### Format:
```json
[
  {
    "email": "user@outlook.com",
    "password": "yourpassword",
    "totpSecret": "",
    "recoveryEmail": "",
    "geoLocale": "auto",
    "langCode": "en",
    "proxy": {
      "proxyAxios": false,
      "url": "",
      "port": 0,
      "username": "",
      "password": ""
    },
    "saveFingerprint": {
      "mobile": false,
      "desktop": false
    }
  }
]
```

> **Using proxy?** Fill in `url`, `port`, `username`, `password` under `proxy`.
> **Using 2FA?** Add `totpSecret` — get it from Microsoft Security > Authenticator app > "Enter code manually".

In [ ]:
# @title Enter account list
import json
import os

ACCOUNTS_JSON = '''[
  {
    "email": "your_email@outlook.com",
    "password": "your_password",
    "totpSecret": "",
    "recoveryEmail": "",
    "geoLocale": "auto",
    "langCode": "en",
    "proxy": {
      "proxyAxios": false,
      "url": "",
      "port": 0,
      "username": "",
      "password": ""
    },
    "saveFingerprint": {
      "mobile": false,
      "desktop": false
    }
  }
]'''  # @param {type:"string"}

# Parse and validate
try:
    accounts = json.loads(ACCOUNTS_JSON)
    if not isinstance(accounts, list):
        raise ValueError("Data must be a JSON array (starts with [ and ends with ])")
    for i, acc in enumerate(accounts):
        if not acc.get("email") or not acc.get("password"):
            raise ValueError(f"Account {i+1}: email and password are required")
    print(f"OK: {len(accounts)} account(s) configured")
    for i, acc in enumerate(accounts):
        proxy_status = "with proxy" if acc.get("proxy", {}).get("url") else "no proxy"
        totp_status = "2FA" if acc.get("totpSecret") else "no 2FA"
        print(f"  {i+1}. {acc['email']} ({proxy_status}, {totp_status})")
except json.JSONDecodeError as e:
    print(f"ERROR: Invalid JSON: {e}")
    print("Check brackets, commas, and quotes!")
    raise

# Save accounts.json
accounts_path = '/content/AutoRewardPlus/src/accounts.json'
with open(accounts_path, 'w') as f:
    json.dump(accounts, f, indent=4)
print(f"\nSaved to {accounts_path}")

In [ ]:
# @title Configure script settings
import json

PROJECT = '/content/AutoRewardPlus'

with open(f'{PROJECT}/src/config.example.json') as f:
    cfg = json.load(f)

# --- Colab-optimized settings ---
cfg['headless'] = True
cfg['clusters'] = 1
cfg['debugLogs'] = True
cfg['globalTimeout'] = '30sec'
cfg['errorDiagnostics'] = True

# Search settings (faster for Colab)
cfg['searchSettings']['searchDelay']['min'] = '10sec'
cfg['searchSettings']['searchDelay']['max'] = '20sec'
cfg['searchSettings']['searchResultVisitTime'] = '5sec'
cfg['searchSettings']['parallelSearching'] = True

# Workers — toggle as needed
# @markdown ### Enable/disable workers:
cfg['workers']['doDesktopSearch'] = True   # @param {type:"boolean"}
cfg['workers']['doMobileSearch'] = True    # @param {type:"boolean"}
cfg['workers']['doDailySet'] = True        # @param {type:"boolean"}
cfg['workers']['doMorePromotions'] = True  # @param {type:"boolean"}
cfg['workers']['doPunchCards'] = True      # @param {type:"boolean"}
cfg['workers']['doAppPromotions'] = True   # @param {type:"boolean"}
cfg['workers']['doSpecialPromotions'] = True # @param {type:"boolean"}
cfg['workers']['doDailyCheckIn'] = True    # @param {type:"boolean"}
cfg['workers']['doReadToEarn'] = True      # @param {type:"boolean"}

with open(f'{PROJECT}/src/config.json', 'w') as f:
    json.dump(cfg, f, indent=4)

print("OK: Config saved (headless mode)")
print()
for k, v in cfg['workers'].items():
    print(f"  {k:24s} = {v}")

## Step 5: Patch Chromium flags + Network check

Add required flags for headless Linux environment and verify connectivity.

In [ ]:
# @title Patch Browser.ts for Colab + Check network
import re
import os
import urllib.request
import ssl

PROJECT = '/content/AutoRewardPlus'
BROWSER_TS = f"{PROJECT}/src/browser/Browser.ts"

# --- Patch Chromium launch args ---
print("=== Patching Chromium flags ===")

with open(BROWSER_TS, 'r', encoding='utf-8') as f:
    content = f.read()

COLAB_ARGS = [
    '--disable-gpu',
    '--disable-dev-shm-usage',
    '--disable-software-rasterizer',
    '--no-proxy-server',
]

missing_args = [a for a in COLAB_ARGS if a not in content]

if missing_args:
    # Find the BROWSER_ARGS array closing and insert before '] as const'
    # Match the last arg line before '] as const'
    pattern = r"('--disable-save-password-bubble')\s*\n(\s*\] as const)"
    insert_lines = '\n'.join(f"        '{arg}'," for arg in missing_args)
    replacement = f"\\1,\n{insert_lines}\n\\2"
    new_content = re.sub(pattern, replacement, content)

    if new_content != content:
        with open(BROWSER_TS, 'w', encoding='utf-8') as f:
            f.write(new_content)
        print(f"  Added {len(missing_args)} flags: {', '.join(missing_args)}")
    else:
        print("  WARNING: Could not find insertion point, adding manually...")
        # Fallback: replace the closing of the array
        content = content.replace(
            "'--disable-save-password-bubble'\n    ] as const",
            "'--disable-save-password-bubble',\n" +
            '\n'.join(f"        '{arg}'," for arg in missing_args) +
            "\n    ] as const"
        )
        with open(BROWSER_TS, 'w', encoding='utf-8') as f:
            f.write(content)
        print(f"  Added {len(missing_args)} flags (fallback method)")
else:
    print("  All Colab flags already present")

# --- Unset proxy env vars that interfere with Chromium ---
print("\n=== Checking proxy environment ===")
proxy_vars = ['http_proxy', 'https_proxy', 'HTTP_PROXY', 'HTTPS_PROXY', 'all_proxy', 'ALL_PROXY']
found_proxy = False
for var in proxy_vars:
    if os.environ.get(var):
        print(f"  Unsetting {var}={os.environ[var]}")
        del os.environ[var]
        found_proxy = True
if not found_proxy:
    print("  No proxy env vars found (good)")

# --- Network connectivity check ---
print("\n=== Checking network connectivity ===")
ssl._create_default_https_context = ssl._create_unverified_context

urls = [
    ("Microsoft Rewards", "https://rewards.bing.com"),
    ("Bing", "https://www.bing.com"),
]

all_ok = True
for name, url in urls:
    try:
        req = urllib.request.Request(url,
            headers={'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36'})
        resp = urllib.request.urlopen(req, timeout=10)
        print(f"  OK: {name} ({resp.status})")
    except Exception as e:
        print(f"  FAIL: {name} — {e}")
        all_ok = False

print()
if all_ok:
    print("Network OK, ready to build and run!")
else:
    print("WARNING: Network issues detected. Try: Runtime > Disconnect and delete runtime, then re-run.")

## Step 6: Build TypeScript

In [ ]:
# @title Build TypeScript
import subprocess
import shutil
import os
import glob

PROJECT = '/content/AutoRewardPlus'

print('=== Building TypeScript ===')
result = subprocess.run('npx tsc', shell=True, capture_output=True, text=True, cwd=PROJECT)

if result.returncode != 0:
    print('Build FAILED:')
    if result.stdout:
        print(result.stdout[-2000:])
    if result.stderr:
        print(result.stderr[-2000:])
    raise RuntimeError("TypeScript build failed")

print('Build OK')

# Copy JSON config files to dist/
for f in ['accounts.json', 'config.json']:
    src = os.path.join(PROJECT, 'src', f)
    dst = os.path.join(PROJECT, 'dist', f)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'  Copied {f} -> dist/')
    else:
        print(f'  WARNING: {f} not found in src/')

# Copy search query JSON files
for src_file in glob.glob(os.path.join(PROJECT, 'src', 'functions', '*.json')):
    dst_dir = os.path.join(PROJECT, 'dist', 'functions')
    os.makedirs(dst_dir, exist_ok=True)
    shutil.copy2(src_file, dst_dir)
    print(f'  Copied {os.path.basename(src_file)} -> dist/functions/')

print('\nReady to run!')

## Step 7: Test Chromium

Quick sanity check that Chromium can launch and navigate.

In [ ]:
# @title Test Chromium launch + navigate
import subprocess
import os

PROJECT = '/content/AutoRewardPlus'

test_code = r'''
const { chromium } = require('patchright');

(async () => {
  const browser = await chromium.launch({
    headless: true,
    args: [
      '--no-sandbox',
      '--disable-gpu',
      '--disable-dev-shm-usage',
      '--disable-software-rasterizer',
      '--no-proxy-server',
      '--ignore-certificate-errors',
      '--disable-setuid-sandbox',
    ]
  });

  const page = await browser.newPage();
  console.log('OK: Browser launched');

  try {
    await page.goto('https://www.bing.com', {
      waitUntil: 'domcontentloaded',
      timeout: 20000
    });
    console.log('OK: Navigated to ' + page.url());
    const title = await page.title();
    console.log('OK: Page title = ' + title);
  } catch (err) {
    console.log('FAIL: Navigation error - ' + err.message);
    process.exit(1);
  }

  await browser.close();
  console.log('OK: Browser closed cleanly');
})().catch(err => { console.log('FATAL: ' + err.message); process.exit(1); });
'''

test_file = os.path.join(PROJECT, '_test_browser.js')
with open(test_file, 'w') as f:
    f.write(test_code)

print("Testing Chromium...")
result = subprocess.run(['node', test_file],
    capture_output=True, text=True, timeout=60, cwd=PROJECT)
print(result.stdout)
if result.returncode != 0:
    if result.stderr:
        print('STDERR:', result.stderr[-800:])
    raise RuntimeError("Chromium test failed")

os.remove(test_file)

## Step 8: Run the bot

This will take ~15-30 minutes per account. Watch the logs below.

In [ ]:
# @title Run AutoRewardPlus
import subprocess
import sys

PROJECT = '/content/AutoRewardPlus'

print("=" * 60)
print("  AutoRewardPlus — Starting...")
print("=" * 60)
print()

result = subprocess.run(
    ['node', 'dist/index.js'],
    cwd=PROJECT,
    stdout=sys.stdout,
    stderr=sys.stderr,
    text=True
)

print()
print("=" * 60)
if result.returncode == 0:
    print("  Bot finished successfully!")
else:
    print(f"  Bot exited with code {result.returncode}")
print("=" * 60)

---
## Troubleshooting

| Error | Cause | Fix |
|---|---|---|
| `chromewebdata/` error loop | Missing Chromium flags | Re-run Step 5 (Patch) before build |
| `Cannot find module 'patchright'` | Missing node_modules | Re-run Step 3 |
| `Browser not found` | Missing Chromium | Re-run Step 3 (Chromium install) |
| `TimeoutError` | Slow network or Microsoft blocking | Increase `globalTimeout` in Step 4 config |
| `Login failed` | Wrong credentials or needs 2FA | Check Step 4, add `totpSecret` if using 2FA |
| `Error: read ECONNRESET` | Unstable proxy | Check proxy settings in account config |
| Session lost | Colab runtime reset | Normal behavior — re-run all cells from start |

**Tips:**
- Enable `debugLogs: true` in config for detailed output
- If login fails, clear sessions: `!rm -rf /content/AutoRewardPlus/dist/browser/sessions`
- Use **Runtime > Restart and run all** to start fresh